# AutoStop: проверка агрегата последних 100 закрытых ЗН

Ноутбук работает только с обезличенным агрегатом `data/private_knowledge/service_pricing_experience.json`. Сырые заказ-наряды, клиенты, VIN, госномера и платежи не загружаются.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
snapshot_path = root / "data/private_knowledge/service_pricing_experience.json"
snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
assert snapshot["privacy"]["aggregate_only"] is True
assert snapshot["privacy"]["contains_order_ids"] is False
assert snapshot["scope"]["selected_closed_orders"] == 100
snapshot["scope"]

In [ ]:
quality = snapshot["data_quality"]
coverage = {
    "valid_work_row_share": quality["work_rows_valid"] / quality["work_rows_total"],
    "article_material_row_share": quality["material_rows_valid_for_article_reference"] / quality["material_rows_total"],
    "orders_without_work_rows": quality["orders_without_work_rows"],
    "orders_without_material_rows": quality["orders_without_material_rows"],
}
coverage

In [ ]:
reusable = [row for row in snapshot["labor_baselines"] if row["sample_count"] >= 3]
top = [
    {
        "operation": row["operation_name"],
        "n": row["sample_count"],
        "median_rub": row["median_rub"],
        "p25_rub": row["p25_rub"],
        "p75_rub": row["p75_rub"],
        "latest": row["latest_closed_date"],
    }
    for row in reusable[:15]
]
assert all(row["n"] >= 3 for row in top)
top

In [ ]:
summary = {
    "labor_operation_groups": len(snapshot["labor_baselines"]),
    "reusable_labor_baselines": len(reusable),
    "article_price_references": len(snapshot["part_price_references"]),
    "high_confidence_rule": snapshot["decision_policy"]["high_confidence_requires_independent_source_families"],
}
assert summary["high_confidence_rule"] == 3
summary